# Full-bank MERT330M WindowMIL · H100

16,042개 전체 자료를 연결된 원곡·화자·프롬프트 단위로 train 12,826 / val 3,216으로 분할했습니다. HTDemucs 기본 모델, MERT330M layer 12, WindowMIL, DF-Arena, PANNs 구조를 유지합니다. MERT·음원 분리·PANNs는 고정하고 WindowMIL과 기존 DF-Arena fc5를 학습합니다.

새 검증 자료에는 과거 로컬 평가 자료가 포함됩니다. 독립 테스트 점수가 아닙니다. 시간 제한은 없으며 검증 EER로 체크포인트를 선택합니다. GPU는 기존 H100 작업 공간을 재사용합니다.


In [ ]:
from pathlib import Path
import json, subprocess, sys
ROOT = Path.cwd()
assert (ROOT / 'full_bank_ops.py').exists(), '프로젝트 루트에서 실행하세요.'
summary = json.loads((ROOT/'results/full_bank_mil_8020_20260918/split_summary.json').read_text())
{k: summary[k] for k in ('admitted','counts','source_counts')}


## 업로드 및 실행

첫 셀은 모든 원본의 해시를 검증해 업로드합니다. 다음 셀은 업로드 완료·원격 파일·8:2 분할·실행 코드의 일치를 확인합니다. 마지막 셀은 유료 H100을 시작합니다. 이미 실행 중인 작업 공간에서는 중복 시작을 거부합니다.


In [ ]:
subprocess.run([sys.executable, 'full_bank_upload.py', ], cwd=ROOT, check=True)


In [ ]:
subprocess.run([sys.executable, 'full_bank_ops.py', 'prepare'], cwd=ROOT, check=True)


In [ ]:
subprocess.run([sys.executable, 'full_bank_ops.py', 'launch'], cwd=ROOT, check=True)


## 원격 실행 과정

전체 길이의 모든 윈도를 추출합니다. MERT batch는 16/32/64/128/256 실측 중 VRAM 80% 이내이며 최고 처리량의 90% 이상인 가장 큰 값을 선택합니다. 분류기 유효 batch는 256이고 메모리에 따라 누적합니다. 각 epoch에 모든 학습 행을 한 번씩 사용하며 seed 41·42·43의 확률을 평균합니다. 최대 50 epoch, 검증 개선 없는 5 epoch에서 수렴 종료합니다. 5분마다 상태를 확인하고 결과 다운로드 및 해시 확인 후 GPU를 중지해야 합니다.
